In [1]:
import pandas as pd
from pathlib import Path

INPUT_DIR = Path('data/validated')
OUTPUT_DIR = Path('data/enriched')
REPORT_DIR = Path('output/reports')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

KEY = ['employee_id', 'attendance_date']
# Only existing Tanggal fields are eligible. Extend after business validation.
CANDIDATE_COLUMNS = ['attendance_type', 'description', 'holiday_type']


def normalize_key(df, name):
    for col in KEY:
        if col not in df.columns:
            raise ValueError(f'{name}: missing {col}')
    df = df.copy()
    df['employee_id'] = df['employee_id'].astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    # Do not overwrite attendance_date; construct a temporary matching key.
    dates = df['attendance_date']
    if not pd.api.types.is_datetime64_any_dtype(dates):
        raise TypeError(f'{name}: attendance_date must already be datetime, got {dates.dtype}. Fix upstream; no automatic conversion.')
    if dates.dropna().dt.year.eq(1970).any():
        raise ValueError(f'{name}: suspicious 1970 dates; fix upstream before enrichment')
    df['_match_date'] = dates.dt.normalize()
    df['_match_employee'] = df['employee_id']
    return df


def empty_mask(s):
    return s.isna() | s.astype('string').str.strip().eq('').fillna(False)


def run():
    tanggal = normalize_key(pd.read_excel(INPUT_DIR / 'tanggal_validated.xlsx', dtype={'employee_id':'string'}), 'tanggal')
    cuti = normalize_key(pd.read_excel(INPUT_DIR / 'cuti_validated.xlsx', dtype={'employee_id':'string'}), 'cuti')
    join_keys = ['_match_employee', '_match_date']
    for name, df in [('tanggal', tanggal), ('cuti', cuti)]:
        if df[join_keys].isna().any(axis=1).any():
            raise ValueError(f'{name}: missing match keys; resolve or separate these rows first')
    if cuti.duplicated(join_keys).any():
        duplicates = cuti.loc[cuti.duplicated(join_keys, keep=False)]
        duplicates.to_excel(REPORT_DIR / 'cuti_duplicate_keys_review.xlsx', index=False)
        raise ValueError('Cuti has duplicate employee-date keys; see cuti_duplicate_keys_review.xlsx')

    shared = [c for c in CANDIDATE_COLUMNS if c in tanggal.columns and c in cuti.columns]
    supplemental = cuti[join_keys + shared].rename(columns={c: f'_cuti_{c}' for c in shared})
    merged = tanggal.merge(supplemental, on=join_keys, how='left', validate='many_to_one', indicator=True, sort=False)
    if len(merged) != len(tanggal):
        raise AssertionError('Enrichment changed row count')

    # Provenance reflects the records contributing to each final row.
    merged['data_source'] = merged['_merge'].map({
        'both': 'tanggal + cuti',
        'left_only': 'tanggal'
    }).astype('string')
    merged['has_conflict'] = False

    conflicts = []
    stats = []
    for col in shared:
        src = merged[f'_cuti_{col}']
        dest = merged[col]
        matched = merged['_merge'].eq('both')
        fill = matched & empty_mask(dest) & ~empty_mask(src)
        conflict = matched & ~empty_mask(dest) & ~empty_mask(src) & dest.astype('string').str.strip().ne(src.astype('string').str.strip()).fillna(False)
        merged.loc[conflict, 'has_conflict'] = True
        if conflict.any():
            c = merged.loc[conflict, join_keys + [col, f'_cuti_{col}']].copy()
            c.insert(0, 'column', col)
            conflicts.append(c)
        merged.loc[fill, col] = src.loc[fill]
        stats.append({'column': col, 'filled': int(fill.sum()), 'conflicts': int(conflict.sum())})

    cuti_only = cuti.merge(tanggal[join_keys].drop_duplicates(), on=join_keys, how='left', indicator=True)
    cuti_only = cuti_only.loc[cuti_only['_merge'].eq('left_only')].drop(columns=['_merge', '_match_employee', '_match_date'])
    cuti_only.to_excel(REPORT_DIR / 'cuti_unmatched.xlsx', index=False)
    conflict_columns = ['column', '_match_employee', '_match_date'] + [c for col in shared for c in (col, f'_cuti_{col}')]
    conflict_report = (
        pd.concat(conflicts, ignore_index=True)
        if conflicts else pd.DataFrame(columns=conflict_columns)
    )
    conflict_report.to_excel(REPORT_DIR / 'enrichment_conflicts.xlsx', index=False)
    pd.DataFrame(stats).to_excel(REPORT_DIR / 'enrichment_summary.xlsx', index=False)

    matched_count = int(merged['_merge'].eq('both').sum())
    result = merged.drop(columns=join_keys + ['_merge'] + [f'_cuti_{c}' for c in shared])
    assert list(result.columns) == [c for c in tanggal.columns if c not in join_keys] + ['data_source', 'has_conflict']
    assert len(result) == len(tanggal)
    assert result.loc[result['has_conflict'], 'data_source'].eq('tanggal + cuti').all()
    output_path = OUTPUT_DIR / 'tanggal_enriched.xlsx'
    result.to_excel(output_path, index=False)
    print('Tanggal input rows:', len(tanggal))
    print('Tanggal output rows:', len(result))
    print('Matched Tanggal rows:', matched_count)
    print('Cuti-only keys:', len(cuti_only))
    print('Conflicting attendance rows:', int(result['has_conflict'].sum()))
    print('Source counts:')
    print(result['data_source'].value_counts(dropna=False).to_string())
    print('Updated columns:', shared)
    print(pd.DataFrame(stats).to_string(index=False))
    print('Output:', output_path)

if __name__ == '__main__':
    run()


Tanggal input rows: 171605
Tanggal output rows: 171605
Matched Tanggal rows: 24450
Cuti-only keys: 5961
Conflicting attendance rows: 24
Source counts:
data_source
tanggal           147155
tanggal + cuti     24450
Updated columns: ['attendance_type']
         column  filled  conflicts
attendance_type     334         24
Output: data\enriched\tanggal_enriched.xlsx
